In [130]:
import xml.etree.ElementTree as ET
from pathlib import Path
import pandas as pd
from form990_parser import (
    parse_header,
    parse_return,
    get_org_type,
    get_return_type,
    NAMESPACE
)

In [114]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [115]:
xml_data_directory = Path(r'xml')

In [4]:
header_data = []
return_data = []
employee_data = []
contractor_data = []
expense_data = []
balance_sheet_data = []

for year_dir in xml_data_directory.iterdir():
    for period_dir in year_dir.iterdir():
        print(period_dir)
        counter = 0
        files_skipped = 0
        for xml_file in period_dir.glob('*.xml'):
            # Skip certain orgs and form types
            root = ET.parse(xml_file).getroot()
            org_type = get_org_type(root)
            if org_type not in ['501c3', '501c4', '527']:
                files_skipped += 1
                continue 

            if counter >= 50:
                break
            counter += 1

            # * Header
            header_content = parse_header(root, xml_file, org_type)
            header_data.append(header_content)

            # * Return
            if parse_return(root, 'tbd') is None:
                continue # Some files pertain to other types of returns (e.g., 990-T)
            return_content, employee_content, contractor_content, expense_content, balance_sheet_content = parse_return(root, 'tbd')
            return_data.append(return_content)
            
            # * Employee
            employee_data += employee_content
            
            # * Contractor
            contractor_data += contractor_content

            # * Expenses
            expense_data += expense_content
            
            # * Balance Sheet
            balance_sheet_data += balance_sheet_content
        # print(counter) # Uncomment when you want to know the actual number of files
        print(f"# Files skipped: {files_skipped}")

xml\2020\2020_1A
# Files skipped: 0
xml\2020\2020_3A
# Files skipped: 11
xml\2024\2024_01A
# Files skipped: 69
xml\2024\2024_02A
# Files skipped: 93
xml\2025\2025_01A
# Files skipped: 113
xml\2025\2025_02A
# Files skipped: 116
xml\2025\2025_03A
# Files skipped: 193
xml\2025\2025_04A
# Files skipped: 191
xml\2025\2025_05A
# Files skipped: 294
xml\2025\2025_05B
# Files skipped: 12
xml\2025\2025_06A
# Files skipped: 142
xml\2025\2025_07A
# Files skipped: 248
xml\2025\2025_08A
# Files skipped: 135
xml\2025\2025_09A
# Files skipped: 139
xml\2025\2025_10A
# Files skipped: 162
xml\2025\2025_11A
# Files skipped: 179
xml\2025\2025_11B
# Files skipped: 12
xml\2025\2025_11C
# Files skipped: 327
xml\2025\2025_11D
# Files skipped: 136
xml\2025\2025_12A
# Files skipped: 112
xml\2026\2026_1A
# Files skipped: 112
xml\2026\2026_2A
# Files skipped: 79
xml\2026\2026_3A
# Files skipped: 178
xml\2026\2026_4A
# Files skipped: 177


## Dataframes

In [5]:
pd.DataFrame(header_data)

,FilerEIN,TaxPeriodBeginDate,TaxPeriodEndDate,TaxYear,ReturnType,FilerBusinessNameLine1Txt,FilerBusinessNameLine2Txt,FilerPhone,FilerAddressLine1Txt,FilerCity,...,PreparerCity,PreparerStateCode,PreparerZipCode,BusinessOfficerName,BusinessOfficerTitle,BusinessOfficerPhone,BusinessOfficerSignatureDate,org_type,file_name,NumOfficers
0,742044647,2018-07-01,2019-06-30,2018,990,NATIONAL JEWISH HEALTH,NaN,3033884461,1400 JACKSON STREET,DENVER,...,NaN,NaN,NaN,Christine Forkner,Chief Financial Officer,3033981004,2020-06-02,501c3,xml\2020\2020_3A\202011559349300711_public.xml,1
1,581416325,2019-01-01,2019-12-31,2019,990,HAWTREE VOLUNTEER FIRE DEPARTMENT INC,NaN,2522573288,PO BOX 116,WISE,...,HENDERSON,NC,27536,STEVE BARNEY,CHIEF,2524325386,2020-05-27,501c3,xml\2020\2020_3A\202011559349300716_public.xml,1
2,223139262,2018-07-01,2019-06-30,2018,990,RURAL DEVELOPMENT INC,NaN,4138639781,241 MILLERS FALLS ROAD,TURNERS FALLS,...,WESTBOROUGH,MA,01581,GINA GOVONI,EXECUTIVE DIRECTOR,4138639781,2020-05-06,501c3,xml\2020\2020_3A\202011559349300726_public.xml,1
3,363244264,2018-07-01,2019-06-30,2018,990,WILL COUNTY GOVERNMENTAL LEAGUE,NaN,8152547700,15905 S FREDERICK STREET Suite 107,PLAINFIELD,...,Oakbrook Terrace,IL,601815209,JOHN NOAK,PRESIDENT,8157293535,2020-05-15,501c4,xml\2020\2020_3A\202011559349300731_public.xml,1
4,043316917,2019-01-01,2019-12-31,2019,990,I'M STILL HERE FOUNDATION INC,NaN,7815690229,10 TOWER OFFICE PARK NO 317,WOBURN,...,BOSTON,MA,02109,JACQUELINE VISCHER,TREASURER,7815690229,2020-05-21,501c3,xml\2020\2020_3A\202011559349300736_public.xml,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1145,341577595,2024-07-01,2025-06-30,2024,990,STARK STATE COLLEGE FOUNDATION,NaN,3304946170,6200 Frank Ave NW,North Canton,...,NaN,NaN,NaN,Joseph Richards,Comptroller,3304946170,2026-03-05,501c3,xml\2026\2026_4A\202600869349300520_public.xml,1
1146,721362424,2025-01-01,2025-12-31,2025,990,NORCO CIVIC ASSOCIATION,NaN,9857649696,PO BOX 22,NORCO,...,NORCO,LA,70079,CLARENCE DUPEPE,TREASURER,9857649696,2026-03-24,501c4,xml\2026\2026_4A\202600869349300525_public.xml,1
1147,980142381,2025-01-01,2025-12-31,2025,990,BETHSHEAN MEXICO MISSION INC,NaN,7062555330,PO BOX 7391,ATHENS,...,HARTWELL,GA,30643,DAN GILLAND,SEC/TREASURER,NaN,2026-03-12,501c3,xml\2026\2026_4A\202600869349300535_public.xml,1
1148,631288598,2025-07-01,2025-09-30,2025,990,C A S A OF FLORENCE LAUDERDALE,NaN,2567650041,102 JACKSON COURT,SHEFFIELD,...,FLORENCE,AL,35630,HEATHER BEGLEY,DIRECTOR,2567650041,2026-03-11,501c3,xml\2026\2026_4A\202600869349300540_public.xml,1


In [6]:
pd.DataFrame(return_data).head()

,PrincipalOfficer,PrincipalOfficerAddress,PrincipalOfficerCity,PrincipalOfficerStateCode,PrincipalOfficerZipCode,GrossReceiptsAmt,ActivityOrMission,MissionDesc,FormationYear,VotingMembersGoverningBodyCnt,...,RelatedOrganizationsAmt,GovernmentGrantsAmt,AllOtherContributionsAmt,NoncashContributionsAmt,TotalContributionsAmt,TotalProgramServiceRevenueAmt,TotalRevenueColumnAmt,RelatedOrExemptFuncIncomeAmt,UnrelatedBusinessRevenueAmt,ExclusionAmt
0,Christine Forkner,1400 Jackson Street,Denver,CO,80206,326863373,National Jewish Health's mission since 1899 is...,National Jewish Health's mission since 1899 is...,1978,44,...,0,60946782,29837556,3056144,96842809,192033726,298202040,187160868,4759037,9439326
1,STEVE BARNEY,PO BOX 116,WISE,NC,27594,107616,TO PROVIDE FIRE DEPARTMENT SERVICES FOR THE CO...,TO PROVIDE FIRE DEPARTMENT SERVICES FOR THE CO...,1981,9,...,NaN,15249,75692,NaN,90941,NaN,103997,3630,0,9426
2,GINA GOVONI,241 MILLERS FALLS ROAD,TURNERS FALLS,MA,01376,301633,"IT IS THE MISSION OF RURAL DEVELOPMENT, INC. (...",IT IS THE MISSION OF RDI TO ADVANCE THE RIGHT ...,1991,8,...,NaN,NaN,6470,NaN,6470,292417,301633,294917,0,246
3,JOHN NOAK,15905 S FREDRICK STREET 107,PLAINFIELD,IL,60586,740025,GOVERNMENT SERVICE,THE WILL COUNTY GOVERNMENTAL LEAGUE IS ORGANIZ...,1983,9,...,NaN,152971,NaN,NaN,152971,472601,664134,472601,NaN,38562
4,JACQUELINE VISCHER,10 TOWER OFFICE PARK NO 317,WOBURN,MA,01801,18632,TO BRING ABOUT COMMUNITY TRANSFORMATION FOR TH...,TO BRING ABOUT COMMUNITY TRANSFORMATION FOR TH...,1995,6,...,NaN,NaN,12724,NaN,12724,5908,18632,5908,0,0


In [7]:
pd.DataFrame(employee_data)

,EIN,PersonNm,TitleTxt,AverageHoursPerWeekRt,AverageHoursPerWeekRltdOrgRt,IndividualTrusteeOrDirectorInd,ReportableCompFromOrgAmt,ReportableCompFromRltdOrgAmt,OtherCompensationAmt,OfficerInd,KeyEmployeeInd,HighestCompensatedEmployeeInd,FormerOfcrDirectorTrusteeInd,BusinessName,InstitutionalTrusteeInd
0,tbd,Jandel Allen-Davis MD,"Member, BOD",2,0,X,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN
1,tbd,Sue Allon,"Member, BOD",2,0,X,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN
2,tbd,Steve Arent,"Lifetime Member, BOD",0,0,X,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN
3,tbd,Richard Baer,"Member, BOD",2,0,X,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN
4,tbd,Geoffrey Barker,"Member, BOD",2,0,X,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
12101,tbd,MARK WOOD,PRESIDENT,NaN,NaN,NaN,0,0,0,X,NaN,NaN,NaN,NaN,NaN
12102,tbd,STEPHEN BARNARD,PRESIDENT,1.00,NaN,X,0,0,0,X,NaN,NaN,NaN,NaN,NaN
12103,tbd,ANITA LEMOS,V.P.,1.00,NaN,X,0,0,0,X,NaN,NaN,NaN,NaN,NaN
12104,tbd,BRYAN GILES,TREASURER,1.00,NaN,X,0,0,0,X,NaN,NaN,NaN,NaN,NaN


In [8]:
pd.DataFrame(contractor_data)

,EIN,BusinessNameLine1Txt,AddressLine1Txt,AddressLine2Txt,CityNm,StateAbbreviationCd,ZIPCd,ServicesDesc,CompensationAmt,PersonNm,CountryCd,BusinessNameLine2Txt,ProvinceOrStateNm,ForeignPostalCd,ContractorAddress
0,tbd,Dimassimo,220 E 23rd Street,2nd Floor,New York,NY,10010,Advertising and Professional Fees,2203123,NaN,NaN,NaN,NaN,NaN,NaN
1,tbd,Siemens Medical Solutions USA Inc,51 Valley Stream Pkwy,NaN,Malvern,PA,19355,Equipment Maintenance Contract,1023832,NaN,NaN,NaN,NaN,NaN,NaN
2,tbd,HSS,MAIN,PO BOX 17033,Denver,CO,80217,Security Support,867767,NaN,NaN,NaN,NaN,NaN,NaN
3,tbd,ARUP Laboratories,MAIN,PO Box 27964,Salt Lake City,UT,84127,Lab Services,848779,NaN,NaN,NaN,NaN,NaN,NaN
4,tbd,Mindset Direct,12110 Sunset Hills Rd 600,NaN,Reston,VA,20190,Fundraising Servicses,612348,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
391,tbd,KENNETH MATTUS,1114 MAPLE STREET,NaN,SANTA MONICA,CA,90405,MUSIC PROGRAM SUPPORT,127140,NaN,NaN,NaN,NaN,NaN,NaN
392,tbd,Tuzinsky Consulting LLC,13741 Labelle St,NaN,Oak Park,MI,48237,Consulting,176325,NaN,NaN,NaN,NaN,NaN,NaN
393,tbd,MTAN Solutions LLC,966 Torke Terrace,NaN,Plymouth,MI,48170,Consulting,107250,NaN,NaN,NaN,NaN,NaN,NaN
394,tbd,Financial One Accounting Inc,44744 Helm St,NaN,Plymouth,MI,48170,Accounting Services,101076,NaN,NaN,NaN,NaN,NaN,NaN


In [9]:
pd.DataFrame(expense_data)

,EIN,ExpenseType,TotalAmt,ProgramServicesAmt,ManagementAndGeneralAmt,FundraisingAmt
0,tbd,Grants_DomesticGroup,0,0,NaN,NaN
1,tbd,Grants_DomesticIndividuals,0,0,NaN,NaN
2,tbd,Compensation_CurrentOfficersDirectors,10636915,6002286,4137346,497283
3,tbd,Fees_Lobbying,154123,0,154123,0
4,tbd,Fees_ProfessionalFundraising,300840,NaN,NaN,300840
...,...,...,...,...,...,...
2407,tbd,Payment_TravelForPublicOfficials,0,0,0,0
2408,tbd,Compensation_CurrentOfficersDirectors,74472,36268,38204,NaN
2409,tbd,Grants_DomesticGroup,2076596,2076596,NaN,NaN
2410,tbd,Grants_DomesticIndividuals,2900,2900,NaN,NaN


In [10]:
pd.DataFrame(balance_sheet_data)

,EIN,BalanceSheetSection,BalanceSheetSubsection,TotalAmt,ProgramServicesAmt,BOYAmt,EOYAmt
0,tbd,Assets,PledgesAndGrantsReceivable,0,0,NaN,NaN
1,tbd,Assets,TotalAssets,NaN,NaN,301872000,304229000
2,tbd,Liabilities,TotalLiabilities,NaN,NaN,88079000,77425000
3,tbd,NetAssetsOrFundBalances,NetAssets_Unrestricted,NaN,NaN,71082000,87606000
4,tbd,NetAssetsOrFundBalances,NetAssets_TemporarilyRestricted,NaN,NaN,90519000,85886000
...,...,...,...,...,...,...,...
4091,tbd,Liabilities,TotalLiabilities,NaN,NaN,0,0
4092,tbd,NetAssetsOrFundBalances,NetAssets_NoDonorRestrictions,NaN,NaN,210531,201416
4093,tbd,Assets,PledgesAndGrantsReceivable,82300,82300,NaN,NaN
4094,tbd,Assets,TotalAssets,NaN,NaN,80845,53390


## Search files for instance that has target

In [131]:
NAMESPACE = '{http://www.irs.gov/efile}'
RETURN_HEADER_PATH = f'./{NAMESPACE}ReturnHeader'
PREPARER_FIRM_GROUP_PATH = f'./{NAMESPACE}PreparerFirmGrp'
FILER_PATH = f'./{NAMESPACE}Filer'
RETURN_DATA_PATH = f'./{NAMESPACE}ReturnData'
IRS990_PATH = RETURN_DATA_PATH + f'/{NAMESPACE}IRS990'
SCHEDULEA_PATH = f'./{NAMESPACE}IRS990ScheduleA'
SCHEDULEC_PATH = f'./{NAMESPACE}IRS990ScheduleC'

filed_990T = 'OrganizationFiled990TInd'

target = f'{NAMESPACE}Organization501cTypeText'

for year_dir in xml_data_directory.iterdir():
    for period_dir in year_dir.iterdir():
        print(period_dir)
        counter = 0
        files_skipped = 0
        for xml_file in period_dir.glob('*.xml'):
            # Skip certain orgs and form types
            root = ET.parse(xml_file).getroot()
            return_type_cd = get_return_type(root)
            org_type = get_org_type(root, org_type)
            if org_type not in ['501c3', '501c4', '527']:
                files_skipped += 1
                continue

            if '202011559349300711' in str(xml_file):
                continue

            _990 = root.find(f'.//{target}')
            if _990 is not None:
                print(xml_file)
                raise TypeError
            # target_text = _990.find(target).text
            # if target_text not in ['false', '0', 'o', 'O']:
            #     print(xml_file)
            #     raise TypeError

xml\2019\2019_01A
xml\2019\2019_02A
xml\2019\2019_03A
xml\2019\2019_TEOS_XML_CT1.zip
xml\2019\download990xml_2019_1.zip
xml\2019\download990xml_2019_2.zip
xml\2019\download990xml_2019_3.zip
xml\2019\download990xml_2019_4.zip
xml\2019\download990xml_2019_5.zip
xml\2019\download990xml_2019_6.zip
xml\2019\download990xml_2019_7.zip
xml\2019\download990xml_2019_8.zip
xml\2019\index_2019.csv
xml\2020\2020_02A
xml\2020\2020_03A


KeyboardInterrupt: 

Search files for instance that does NOT have target

In [5]:
NAMESPACE = '{http://www.irs.gov/efile}'
RETURN_HEADER_PATH = f'./{NAMESPACE}ReturnHeader'
PREPARER_FIRM_GROUP_PATH = f'./{NAMESPACE}PreparerFirmGrp'
FILER_PATH = f'./{NAMESPACE}Filer'
RETURN_DATA_PATH = f'./{NAMESPACE}ReturnData'
IRS990_PATH = RETURN_DATA_PATH + f'/{NAMESPACE}IRS990'

target = f'{NAMESPACE}RelatedOrgSect527OrgInd'

for year_dir in xml_data_directory.iterdir():
    for period_dir in year_dir.iterdir():
        print(period_dir)
        counter = 0
        files_skipped = 0
        for xml_file in period_dir.glob('*.xml'):
            # Skip certain orgs and form types
            root = ET.parse(xml_file).getroot()
            org_type = get_org_type(root)
            if org_type not in ['501c3', '501c4', '527']:
                files_skipped += 1
                continue 
            
            if '202011559349301216' in str(xml_file):
                continue
            
            _990 = root.find(IRS990_PATH)
            try:
                _990.find(target).text
            except AttributeError:
                print(xml_file)
                raise AttributeError

xml\2019\2019_01A
xml\2019\2019_01A\201900079349300000_public.xml


AttributeError: 

In [116]:
from construct_reference_matrices import construct_reference_matrices

In [117]:
construct_reference_matrices(xml_data_directory, 'reference_matrices')

Parsed 1921000/1921000 scanned files.
Parsed 1922000/1922000 scanned files.
Parsed 1923000/1923000 scanned files.
Parsed 1924000/1924000 scanned files.
Parsed 1925000/1925000 scanned files.
Parsed 1926000/1926000 scanned files.
Parsed 1927000/1927000 scanned files.
Parsed 1928000/1928000 scanned files.
Parsed 1929000/1929000 scanned files.
Parsed 1930000/1930000 scanned files.
Parsed 1931000/1931000 scanned files.
Parsed 1932000/1932000 scanned files.
Parsed 1933000/1933000 scanned files.
Parsed 1934000/1934000 scanned files.
Parsed 1935000/1935000 scanned files.
Parsed 1936000/1936000 scanned files.
Parsed 1937000/1937000 scanned files.
Parsed 1938000/1938000 scanned files.
Parsed 1939000/1939000 scanned files.
Parsed 1940000/1940000 scanned files.
Parsed 1941000/1941000 scanned files.
Parsed 1942000/1942000 scanned files.
Parsed 1943000/1943000 scanned files.
Parsed 1944000/1944000 scanned files.
Parsed 1945000/1945000 scanned files.
Parsed 1946000/1946000 scanned files.
Parsed 19470

In [122]:
reference_matrix_df = pd.DataFrame()
for reference_matrix_csv in Path('reference_matrices').glob('*.csv'):
    new_reference_matrix_csv = pd.read_csv(
        reference_matrix_csv,
        dtype={'ReturnTypeCd': str}
    )
    reference_matrix_df = pd.concat(
        [reference_matrix_df, new_reference_matrix_csv]
    )
reference_matrix_df.head()

,EIN,Name,OrgType,file,TaxYr,ReturnTypeCd,IRS990,IRS990Size,IRS990ScheduleA,IRS990ScheduleASize,...,ExplanOfLegisPoliticalActvts,ExplanOfLegisPoliticalActvtsSize,AffiliateListing,AffiliateListingSize,AffiliatedGroupAttachment,AffiliatedGroupAttachmentSize,BorrowedFundsElection,BorrowedFundsElectionSize,TaxUnderSection511Statement,TaxUnderSection511StatementSize
0,860996104,ROAD 2 RECOVERY FOUNDATION,501c3,xml\2019\2019_03A\201912829349301351_public.xml,2018,990,1.0,238.0,1.0,14.0,...,0.0,0.0,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN
1,316067780,AMERICAN JERSEY CATTLE CLUB,501c3,xml\2019\2019_03A\201912829349301401_public.xml,2018,990,1.0,182.0,1.0,12.0,...,0.0,0.0,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN
2,391270706,NAMI DANE COUNTY INC,501c3,xml\2019\2019_03A\201912829349301406_public.xml,2018,990,1.0,216.0,1.0,10.0,...,0.0,0.0,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN
3,416038263,MINNESOTA FFA FOUNDATION INC,501c3,xml\2019\2019_03A\201912829349301411_public.xml,2018,990,1.0,269.0,1.0,15.0,...,0.0,0.0,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN
4,272635994,Pima County Community Land Trust,501c3,xml\2019\2019_03A\201912829349301416_public.xml,2018,990,1.0,264.0,1.0,15.0,...,0.0,0.0,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN


In [132]:
num_examples = 1
tax_years = reference_matrix_df.loc[:, 'TaxYr'].unique()
return_types = reference_matrix_df.loc[:, 'ReturnTypeCd'].unique()
org_types = reference_matrix_df.loc[:, 'OrgType'].unique()
cols_to_exclude = ['EIN', 'Name', 'OrgType', 'file', 'TaxYr', 'ReturnTypeCd']
sections = [col for col in reference_matrix_df.columns if col not in cols_to_exclude and 'Size' not in col]
example_df = pd.DataFrame()
for tax_year in tax_years:
    tax_year_mask = reference_matrix_df.loc[:, 'TaxYr'] == tax_year
    # print(tax_year)
    # print(len(reference_matrix_df.loc[tax_year_mask]))
    for return_type in return_types:
        return_type_mask = reference_matrix_df.loc[:, 'ReturnTypeCd'] == return_type
        # print("\t", return_type)
        # print("\t", len(reference_matrix_df.loc[tax_year_mask & return_type_mask]))

        for org_type in org_types:
            org_type_mask = reference_matrix_df.loc[:, 'OrgType'] == org_type
            # print("\t", "\t", org_type)
            # print("\t", "\t", len(reference_matrix_df.loc[tax_year_mask & return_type_mask & org_type_mask]))

            for section in sections:
                section_mask = reference_matrix_df.loc[:, section] == 1
                masks = tax_year_mask & return_type_mask & org_type_mask & section_mask
                # print("\t", "\t", "\t", section)
                # print("\t", "\t", "\t", len(reference_matrix_df.loc[tax_year_mask & return_type_mask & org_type_mask & section_mask]))
                # print("\t", "\t", "\t", len(reference_matrix_df.loc[masks]))
                section_df = reference_matrix_df.loc[masks].sort_values(by=f'{section}Size', ascending=False)
                section_df = section_df.iloc[:num_examples, :]
                # Describe usecase
                situation_str = f'A {org_type} filed a {return_type} containing {section} in {tax_year}.'
                # Using number of rows in section_df in case there are examples, but less than 'num_examples'
                situation_col = pd.Series([situation_str] * len(section_df), name='situation')
                    
                section_df = pd.concat(
                    [situation_col, section_df.reset_index(drop=True)],
                    axis=1
                )
                example_df = pd.concat([example_df, section_df])

In [133]:
example_df.drop_duplicates().to_csv(
    'example_usecase_lookup.csv',
    index=False
)

In [135]:
len(example_df.file.unique())

1044

In [142]:
m = example_df.loc[:, 'file'].str.contains('2019_03A')
example_df.loc[m]

,situation,EIN,Name,OrgType,file,TaxYr,ReturnTypeCd,IRS990,IRS990Size,IRS990ScheduleA,...,ExplanOfLegisPoliticalActvts,ExplanOfLegisPoliticalActvtsSize,AffiliateListing,AffiliateListingSize,AffiliatedGroupAttachment,AffiliatedGroupAttachmentSize,BorrowedFundsElection,BorrowedFundsElectionSize,TaxUnderSection511Statement,TaxUnderSection511StatementSize
0,A 501c3 filed a 990 containing IRS990 in 2018.,136227614,HADASSAH GROUP RETURN,501c3,xml\2019\2019_03A\201913199349306206_public.xml,2018,990,1.0,3652.0,1.0,...,0.0,0.0,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN
0,A 501c3 filed a 990 containing IRS990ScheduleA...,135562401,YOUNG MEN'S CHRISTIAN ASSOCIATION RETIREMENT FUND,501c3,xml\2019\2019_03A\201913159349303926_public.xml,2018,990,1.0,267.0,1.0,...,0.0,0.0,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN
0,A 501c3 filed a 990 containing IRS990ScheduleF...,510198509,TIDES FOUNDATION,501c3,xml\2019\2019_03A\201913189349314251_public.xml,2018,990,1.0,274.0,1.0,...,0.0,0.0,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN
0,A 501c3 filed a 990 containing IRS990ScheduleG...,810421425,ROCKY MOUNTAIN ELK FOUNDATION INC,501c3,xml\2019\2019_03A\201913179349302811_public.xml,2018,990,1.0,368.0,1.0,...,0.0,0.0,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN
0,A 501c3 filed a 990 containing IRS990ScheduleM...,474659654,One Summit Inc,501c3,xml\2019\2019_03A\201912769349301126_public.xml,2018,990,1.0,255.0,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
0,A 501c3 filed a 990EZ containing TransferPrsnl...,43283220,Friends of Gymnastics,501c3,xml\2019\2019_03A\201912879349201101_public.xml,2016,990EZ,0.0,0.0,1.0,...,0.0,0.0,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN
0,A 501c3 filed a 990EZ containing ReasonableCau...,330909316,Pacifica Choral Boosters,501c3,xml\2019\2019_03A\201913199349207821_public.xml,2016,990EZ,0.0,0.0,1.0,...,0.0,0.0,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN
0,A 501c3 filed a 990EZ containing CompensationE...,263828109,Hands Helping Paws Inc,501c3,xml\2019\2019_03A\201913189349201396_public.xml,2016,990EZ,0.0,0.0,1.0,...,0.0,0.0,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN
0,A 501c3 filed a 990EZ containing IRS990Schedul...,900396973,COMMUNITY CARE INC,501c3,xml\2019\2019_03A\201912559349200211_public.xml,2016,990EZ,0.0,0.0,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,NaN


In [134]:
pd.read_csv( 'example_usecase_lookup.csv')

,situation,EIN,Name,OrgType,file,TaxYr,ReturnTypeCd,IRS990,IRS990Size,IRS990ScheduleA,...,ExplanOfLegisPoliticalActvts,ExplanOfLegisPoliticalActvtsSize,AffiliateListing,AffiliateListingSize,AffiliatedGroupAttachment,AffiliatedGroupAttachmentSize,BorrowedFundsElection,BorrowedFundsElectionSize,TaxUnderSection511Statement,TaxUnderSection511StatementSize
0,A 501c3 filed a 990 containing IRS990 in 2018.,136227614,HADASSAH GROUP RETURN,501c3,xml\2019\2019_03A\201913199349306206_public.xml,2018,990,1.0,3652.0,1.0,...,0.0,0.0,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN
1,A 501c3 filed a 990 containing IRS990ScheduleA...,135562401,YOUNG MEN'S CHRISTIAN ASSOCIATION RETIREMENT FUND,501c3,xml\2019\2019_03A\201913159349303926_public.xml,2018,990,1.0,267.0,1.0,...,0.0,0.0,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN
2,A 501c3 filed a 990 containing IRS990ScheduleB...,452318579,CARIBBEAN STUDIES ASSOCIATIONINC,501c3,xml\2021\2021_01A\202101069349300000_public.xml,2018,990,1.0,190.0,1.0,...,0.0,0.0,NaN,NaN,0.0,0.0,0.0,0.0,0.0,0.0
3,A 501c3 filed a 990 containing IRS990ScheduleD...,621577879,OBION COUNTY HABITAT FOR HUMANITY,501c3,xml\2019\2019_01A\201901349349305675_public.xml,2018,990,1.0,189.0,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,NaN
4,A 501c3 filed a 990 containing IRS990ScheduleF...,510198509,TIDES FOUNDATION,501c3,xml\2019\2019_03A\201913189349314251_public.xml,2018,990,1.0,274.0,1.0,...,0.0,0.0,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1120,A 501c4 filed a 990EZ containing TransferPrsnl...,841417032,CARBONDALE AGRICULTURAL HERITAGE FUND,501c4,xml\2026\2026_4A\202631109349200428_public.xml,2025,990EZ,0.0,0.0,0.0,...,0.0,0.0,NaN,NaN,NaN,NaN,0.0,0.0,NaN,NaN
1121,A 501c4 filed a 990EZ containing ReasonableCau...,621754015,AFRO AMERICAN POLICE ASSOCIATION,501c4,xml\2026\2026_4A\202631079349201128_public.xml,2025,990EZ,0.0,0.0,1.0,...,0.0,0.0,NaN,NaN,NaN,NaN,0.0,0.0,NaN,NaN
1122,A 501c4 filed a 990EZ containing CompensationE...,813842947,TEJAS SILHOUETTE SHOOTERS,501c4,xml\2026\2026_3A\202600709349201710_public.xml,2025,990EZ,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,NaN,NaN,0.0,0.0,0.0,0.0
1123,A 501c4 filed a 990EZ containing IRS990Schedul...,822944257,National Vote At Home Coalition,501c4,xml\2026\2026_4A\202640929349200614_public.xml,2025,990EZ,0.0,0.0,0.0,...,0.0,0.0,NaN,NaN,NaN,NaN,0.0,0.0,NaN,NaN


In [ ]:
# filing_content_matrix = pd.DataFrame(data)
# filing_content_matrix.loc[:, 'file'] = filing_content_matrix.loc[:, 'file'].apply(lambda x: '\\'.join(x.split('\\')[2:]))
# filing_content_matrix[filing_content_matrix.isna()] = 0
# cols_to_convert = [col for col in filing_content_matrix.columns if col not in ['EIN', 'Name', 'OrgType', 'file']]
# for col in cols_to_convert:
#     filing_content_matrix[col] = filing_content_matrix[col].astype(int)
# filing_content_matrix.to_csv('filing_content_matrix.csv')

In [ ]:
# filing_content_matrix

,EIN,Name,OrgType,file,TaxYr,IRS990,IRS990Size,IRS990ScheduleA,IRS990ScheduleASize,IRS990ScheduleB,...,IRS990ScheduleH,IRS990ScheduleHSize,AveragingAttachment,AveragingAttachmentSize,AffiliatedGroupSchedule,AffiliatedGroupScheduleSize,AffiliateListing,AffiliateListingSize,AffiliatedGroupAttachment,AffiliatedGroupAttachmentSize
0,222964056,COMMUNITY OPTIONS INC,501c3,2019_01A\201900079349300000_public.xml,2017,1,296,1,15,1,...,0,0,0,0,0,0,0,0,0,0
1,201305421,BPRAPTOR CENTER,501c3,2019_01A\201900079349300005_public.xml,2017,1,206,1,13,1,...,0,0,0,0,0,0,0,0,0,0
2,223839872,DR CLARENCE YORK FOUNDATION,501c3,2019_01A\201900079349300050_public.xml,2017,1,247,1,14,1,...,0,0,0,0,0,0,0,0,0,0
3,020385121,GRANITE STATE ECONOMIC DEVELOPMENT CORP,501c4,2019_01A\201900079349300055_public.xml,2017,1,215,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,273719389,COMMUNITY OPTIONS HOPEWELL INC,501c3,2019_01A\201900079349300100_public.xml,2017,1,235,1,19,0,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
911361,431556293,CIEFA MINISTRY INTERNATIONAL,501c3,2026_4A\202641139349301959_public.xml,2025,1,251,1,22,0,...,0,0,0,0,0,0,0,0,0,0
911362,260637571,ARMS OF CARE INTERNATIONAL INC,501c3,2026_4A\202641139349302004_public.xml,2025,1,176,1,8,0,...,0,0,0,0,0,0,0,0,0,0
911363,382878506,Christ Centered Homes Inc,501c3,2026_4A\202641139349302009_public.xml,2024,1,199,1,9,0,...,0,0,0,0,0,0,0,0,0,0
911364,330416470,HABITAT FOR HUMANITY OF GREATER,501c3,2026_4A\202641139349302054_public.xml,2024,1,278,1,11,1,...,0,0,0,0,0,0,0,0,0,0


In [9]:
filing_content_matrix = pd.read_csv('filing_content_matrix.csv').iloc[:, 1:]
filing_content_matrix#.iloc[filing_content_matrix.EIN == 264486735, :]

,EIN,Name,OrgType,file,TaxYr,IRS990,IRS990Size,IRS990ScheduleA,IRS990ScheduleASize,IRS990ScheduleB,...,IRS990ScheduleH,IRS990ScheduleHSize,AveragingAttachment,AveragingAttachmentSize,AffiliatedGroupSchedule,AffiliatedGroupScheduleSize,AffiliateListing,AffiliateListingSize,AffiliatedGroupAttachment,AffiliatedGroupAttachmentSize
0,222964056,COMMUNITY OPTIONS INC,501c3,2019_01A\201900079349300000_public.xml,2017,1,296,1,15,1,...,0,0,0,0,0,0,0,0,0,0
1,201305421,BPRAPTOR CENTER,501c3,2019_01A\201900079349300005_public.xml,2017,1,206,1,13,1,...,0,0,0,0,0,0,0,0,0,0
2,223839872,DR CLARENCE YORK FOUNDATION,501c3,2019_01A\201900079349300050_public.xml,2017,1,247,1,14,1,...,0,0,0,0,0,0,0,0,0,0
3,20385121,GRANITE STATE ECONOMIC DEVELOPMENT CORP,501c4,2019_01A\201900079349300055_public.xml,2017,1,215,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,273719389,COMMUNITY OPTIONS HOPEWELL INC,501c3,2019_01A\201900079349300100_public.xml,2017,1,235,1,19,0,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
911361,431556293,CIEFA MINISTRY INTERNATIONAL,501c3,2026_4A\202641139349301959_public.xml,2025,1,251,1,22,0,...,0,0,0,0,0,0,0,0,0,0
911362,260637571,ARMS OF CARE INTERNATIONAL INC,501c3,2026_4A\202641139349302004_public.xml,2025,1,176,1,8,0,...,0,0,0,0,0,0,0,0,0,0
911363,382878506,Christ Centered Homes Inc,501c3,2026_4A\202641139349302009_public.xml,2024,1,199,1,9,0,...,0,0,0,0,0,0,0,0,0,0
911364,330416470,HABITAT FOR HUMANITY OF GREATER,501c3,2026_4A\202641139349302054_public.xml,2024,1,278,1,11,1,...,0,0,0,0,0,0,0,0,0,0


In [10]:
filing_content_matrix.loc[filing_content_matrix.OrgType == '527', :]

,EIN,Name,OrgType,file,TaxYr,IRS990,IRS990Size,IRS990ScheduleA,IRS990ScheduleASize,IRS990ScheduleB,...,IRS990ScheduleH,IRS990ScheduleHSize,AveragingAttachment,AveragingAttachmentSize,AffiliatedGroupSchedule,AffiliatedGroupScheduleSize,AffiliateListing,AffiliateListingSize,AffiliatedGroupAttachment,AffiliatedGroupAttachmentSize


In [6]:
for col in filing_content_matrix.columns:
    print(col)

EIN
Name
OrgType
file
TaxYr
IRS990
IRS990ScheduleA
IRS990ScheduleC
IRS990ScheduleD
IRS990ScheduleG
IRS990ScheduleH
IRS990ScheduleJ
IRS990ScheduleK
IRS990ScheduleL
IRS990ScheduleM
IRS990ScheduleO
IRS990ScheduleR
IRS990ScheduleB
IRS990ScheduleI
IRS990ScheduleF
AffiliatedGroupAttachment
IRS990ScheduleE
ReasonableCauseExplanation
IRS990ScheduleN
AveragingAttachment
AffiliatedGroupSchedule
AffiliateListing


In [12]:
filing_type_by_year = {}
matches = []
for year_dir in xml_data_directory.iterdir():
    for period_dir in year_dir.iterdir():
        counter = 0
        period_types = []
        for xml_file in period_dir.glob('*.xml'):
            # Arbitrary limit
            if counter >= 250:
                break
            counter += 1

            root = ET.parse(xml_file).getroot()
            for elem in root.iter():
                if 'return' in elem.tag.lower():
                    # return_type_code = elem.tag
                    matches.append(elem.tag)
            # print(return_type_code)